# G6A 实验启动

在 VS Code 打开本 notebook，右上角选择 Python 内核，然后依次运行单元格。不需要打开 CMD。
计算仍使用原实验 Python 环境，默认 **12 个单线程 worker**。

- `MODE = "run"`：运行/续算本组，完成后自动验收和生成 CSV、MD、PDF 矢量图。
- `MODE = "export"`：已有完整记录时，仅重新汇总、验收与绘图。
- 输出在本实验目录的 `汇总表_CSV`、`结果_MD`、`图`；原始记录在代码目录 `_work`。
- 一次只运行一个实验。中途停止请使用中断按钮，并等待单元格结束；不要直接关闭内核，以免留下子进程。

未执行任何单元格，输出为空。


## 1. 准备环境
仅准备路径和启动函数，不启动实验。

In [ ]:
from pathlib import Path
import os, subprocess
STAGE = 'G6A'
PYTHON = Path(r"C:\Users\Lenovo\Documents\ATLAS\runtime\Scripts\python.exe")
candidates = []
for p in (Path.cwd(), *Path.cwd().parents):
    candidates += [p, p / f"{STAGE}实验_代码", p / f"{STAGE}实验" / f"{STAGE}实验_代码",
                   p / "实验交付" / f"{STAGE}实验" / f"{STAGE}实验_代码"]
candidates.append(Path('C:\\Users\\Lenovo\\Documents\\ChatGPT\\毕设\\实验交付\\G6A实验\\G6A实验_代码'))
CODE_DIR = next((p for p in candidates if p.name == f"{STAGE}实验_代码" and (p / "runner.py").is_file()), None)
assert CODE_DIR is not None, "请设置正确的 CODE_DIR。"
assert PYTHON.is_file(), "请设置原实验环境的 python.exe 路径。"
ENV = os.environ.copy()
for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','BLIS_NUM_THREADS'):
    ENV[k] = '1'
ENV['PYTHONIOENCODING'] = 'utf-8'
ENV['PYTHONUNBUFFERED'] = '1'
ENV['MPLBACKEND'] = 'Agg'

def run_command(arguments):
    # Run the script as a separate Python process for Windows multiprocessing.
    p = subprocess.Popen([str(PYTHON), '-u', *arguments], cwd=str(CODE_DIR), env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace', bufsize=1)
    from collections import deque
    recent = deque(maxlen=25)
    try:
        for line in p.stdout:
            recent.append(line)
            print(line, end='', flush=True)
        rc = p.wait()
        if rc:
            raise RuntimeError(f'程序退出码 {rc}。原始输出末尾：\n' + ''.join(recent))
    except BaseException:
        if p.poll() is None:
            subprocess.run(['taskkill', '/PID', str(p.pid), '/T', '/F'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            p.wait()
        raise
    finally:
        p.stdout.close()

print('实验：', STAGE)
print('代码：', CODE_DIR)
print('输出：', CODE_DIR.parent)


## 2. 选择运行模式

In [ ]:
MODE = 'run'  # 'run': 运行/续算；'export': 仅汇总和绘图
WORKERS = 12
assert MODE in ('run', 'export')
assert WORKERS == 12, '按当前约定使用 12 个单线程 worker。'


## 3. 启动本组
执行下面单元格才会启动实验。运行进度和错误显示在下方。

In [ ]:
arguments = [str(CODE_DIR / 'runner.py'), '--stage', STAGE, '--workers', str(WORKERS)]
if MODE == 'export':
    arguments.append('--export-only')
run_command(arguments)
print('完成。输出目录：', CODE_DIR.parent)
